In [1]:
from abc import ABC


In [2]:
class Tokenizer(ABC):
    """Abstract interface for a tokenizer."""
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError
    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError

In [3]:
# Implementation using comprehension
class CharacterTokenizer(Tokenizer):
    """Represent a string as a sequence of Unicode code points."""
    def encode(self, string: str) -> list[int]:
        return [ord(c) for c in string]
    def decode(self, indices: list[int]) -> str:
        return ''.join(chr(i) for i in indices)

In [4]:
# Implementation using map
class CharacterTokenizer(Tokenizer):
    """Represent a string as a sequence of Unicode code points."""
    def encode(self, string: str) -> list[int]:
        return list(map(ord, string))
    def decode(self, indices: list[int]) -> str:
        return ''.join(map(chr, indices))

In [5]:
x = "hello"
character_tokenizer = CharacterTokenizer()
character_tokenizer.encode(x)

[104, 101, 108, 108, 111]

In [6]:
character_tokenizer.decode([104, 101, 108, 108, 111])

'hello'

In [7]:
class ByteTokenizer(Tokenizer):
    """Represent a string as a sequence of bytes."""
    def encode(self, string: str) -> list[int]:
        return list(string.encode('UTF-8'))
    def decode(self, indices: list[int]) -> str:
        return bytes(indices).decode()

In [8]:
byte_tokenizer = ByteTokenizer()
byte_tokenizer.encode('牛')

[231, 137, 155]

In [9]:
byte_tokenizer.decode([231, 137, 155])

'牛'

In [10]:
from dataclasses import dataclass
@dataclass
class BPETokenizerParams:
    """All you need to specify a BPETokenizer."""
    vocab: dict[int, bytes]     # index -> bytes
    merges: dict[tuple[int, int], int]  # index1,index2 -> new_index

In [11]:
def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:  
    """Return `indices`, but with all instances of `pair` replaced with `new_index`."""
    new_indices = []  
    i = 0  
    while i < len(indices):
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    return new_indices

In [12]:
class BPETokenizer(Tokenizer):
    """BPE tokenizer given a set of merges and a vocabulary."""
    def __init__(self, params: BPETokenizerParams):
        self.params = params
    def encode(self, string: str) -> list[int]:
        indices = list(map(int, string.encode("utf-8")))  
        # Note: this is a very slow implementation
        for pair, new_index in self.params.merges.items():  
            indices = merge(indices, pair, new_index)  
        return indices
    def decode(self, indices: list[int]) -> str:
        bytes_list = list(map(self.params.vocab.get, indices))  
        string = b"".join(bytes_list).decode("utf-8")  
        return string

In [13]:
from collections import defaultdict

def count_adjacent_pairs(indices: list[int]) -> dict[tuple[int, int], int]:
    """Return a dictionary mapping each adjacent pair of tokens in `indices` to the number of times it occurs."""
    counts = defaultdict(int)
    for index1, index2 in zip(indices, indices[1:]):
        counts[(index1, index2)] += 1
    return counts

In [14]:
def train_bpe(string: str, num_merges: int) -> BPETokenizerParams:  
    indices = list(map(int, string.encode("utf-8")))  
    merges: dict[tuple[int, int], int] = {}  # index1, index2 => merged index
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}  # index -> bytes

    for i in range(num_merges):
        # Count the number of occurrences of each pair of tokens
        counts = count_adjacent_pairs(indices)  
        print("counts: ", counts)
        # Find the most common pair
        pair = max(counts, key=counts.get)  
        # Merge that pair
        new_index = 256 + i  
        merges[pair] = new_index  
        vocab[new_index] = vocab[pair[0]] + vocab[pair[1]]  
        indices = merge(indices, pair, new_index)

    return BPETokenizerParams(vocab=vocab, merges=merges)

In [15]:
string = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"  
params = train_bpe(string, num_merges=6)
params.vocab

counts:  defaultdict(<class 'int'>, {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 8, (101, 114): 2, (114, 32): 2, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 101): 3, (101, 115): 9, (115, 116): 9, (116, 32): 8, (32, 110): 6, (110, 101): 6, (101, 119): 6})
counts:  defaultdict(<class 'int'>, {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2, (101, 114): 2, (114, 32): 2, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 256): 3, (256, 116): 9, (116, 32): 8, (32, 110): 6, (110, 101): 6, (101, 119): 6, (119, 256): 6})
counts:  defaultdict(<class 'int'>, {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2, (101, 114): 2, (114, 32): 2, (32, 119): 3, (119, 105): 3, (105, 100): 3, (100, 257): 3, (257, 32): 8, (32, 110): 6, (110, 101): 6, (101, 119): 6, (119, 257): 6})
counts:  defaultdict(<class 'int'>, {(108, 111): 7, (111, 119): 7, (119, 32): 5, (32, 108): 6, (119, 101): 2, (101, 114): 2, (114, 32): 2, (32, 119): 1, (

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [13]:
"low".encode("utf-8")

b'low'

In [8]:
"low".encode("utf-8")

b'low'

In [12]:
list("low".encode("utf-8"))

[108, 111, 119]

In [3]:
"low".encode("utf-8")[0]

108

In [9]:
indices_list = [108, 111, 119]

In [11]:
[bytes([i]) for i in indices_list]

[b'l', b'o', b'w']

In [4]:
bytes([108])

b'l'

In [7]:
b'low'[0:1]

b'l'

In [ ]:
(b' ', b't', b'w', b'i', b'r', b'l', b'e', b'r')

(b't', b'w', b'i', b'r', b'l', b'e', b'r')

In [19]:
"|".join(["w","c","m"])

'w|c|m'

In [20]:
b't' +  b'w'

b'tw'